# Option Pricing with RustQuant (Python)

Python version of `02_option_pricing.ipynb` using the PyO3 bindings.

## Setup

In [ ]:
import math
from RustQuant.instruments import BlackScholesMerton, OptionType
from RustQuant.stochastics import GeometricBrownianMotion

## 1. Black-Scholes-Merton Pricing

In [ ]:
call = BlackScholesMerton(
    underlying_price=100.0, strike_price=100.0, volatility=0.20,
    risk_free_rate=0.05, cost_of_carry=0.05,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)

print("=== European Call Option (ATM) ===")
print(f"Price  = {call.price():.4f}")
print(f"Delta  = {call.delta():.4f}")
print(f"Gamma  = {call.gamma():.4f}")
print(f"Theta  = {call.theta():.4f}")
print(f"Vega   = {call.vega():.4f}")
print(f"Rho    = {call.rho():.4f}")

## 2. Put Option and Put-Call Parity

In [ ]:
put = BlackScholesMerton(
    underlying_price=100.0, strike_price=100.0, volatility=0.20,
    risk_free_rate=0.05, cost_of_carry=0.05,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Put,
)

print(f"Call Price = {call.price():.4f}")
print(f"Put Price  = {put.price():.4f}")
print(f"\nPut-Call Parity: C - P = {call.price() - put.price():.4f}")
print(f"                 S - K*exp(-rT) ≈ {100.0 - 100.0 * math.exp(-0.05):.4f}")

## 3. Implied Volatility

In [ ]:
market_price = call.price()
iv = call.implied_volatility(market_price)
print(f"Market price (from 20% vol): {market_price:.4f}")
print(f"Implied volatility: {iv:.6f} (expected ~0.20)")

## 4. Strike Sensitivity

In [ ]:
print(f"{'Strike':<10} {'Price':<12} {'Delta':<10} {'Gamma':<10}")
print("-" * 42)

for k in range(80, 125, 5):
    opt = BlackScholesMerton(
        underlying_price=100.0, strike_price=float(k), volatility=0.20,
        risk_free_rate=0.05, cost_of_carry=0.05,
        expiry_year=2027, expiry_month=3, expiry_day=22,
        option_type=OptionType.Call,
    )
    print(f"{k:<10} {opt.price():<12.4f} {opt.delta():<10.4f} {opt.gamma():<10.6f}")

## 5. Monte Carlo Option Pricing

In [ ]:
gbm = GeometricBrownianMotion(mu=0.05, sigma=0.20)
traj = gbm.simulate(x0=100.0, t_end=1.0, n_steps=252, n_paths=50_000)

df = math.exp(-0.05 * 1.0)
mc_call = df * sum(max(p[-1] - 100.0, 0) for p in traj.paths) / len(traj.paths)
print(f"MC Call Price:  {mc_call:.4f} (analytic: {call.price():.4f})")

## 6. Exotic Options: Asian (Path-Dependent)

In [ ]:
import statistics

asian_call = df * sum(
    max(statistics.mean(p) - 100.0, 0) for p in traj.paths
) / len(traj.paths)

print(f"Asian Call (arithmetic avg): {asian_call:.4f}")
print(f"Vanilla Call:               {mc_call:.4f}")
print("\nAsian options are cheaper due to averaging reducing volatility.")

## Summary

| Method | Python API |
|--------|------------|
| Black-Scholes-Merton | `BlackScholesMerton(...).price()` |
| Greeks | `.delta()`, `.gamma()`, `.vega()`, `.theta()`, `.rho()` |
| All Greeks | `.greeks()` returns dict |
| Implied Volatility | `.implied_volatility(price)` |
| Monte Carlo | `GeometricBrownianMotion.simulate()` + Python payoff |